# Bài Thực Hành Chương 1, 2, 3: Nền Tảng Python & Lập Trình Hướng Đối Tượng (OOP)

### Mục tiêu học tập (CLO1, CLO3, CLO5):
- **Chương 1**: Thiết lập môi trường Python, kiểm tra Jupyter Notebook.
- **Chương 2**: Cú pháp, kiểu dữ liệu cơ bản (int, float, str, bool), cấu trúc dữ liệu phức hợp (list, tuple, dict, set), cấu trúc rẽ nhánh, vòng lặp, hàm, đọc ghi file và xử lý ngoại lệ (try-except-finally).
- **Chương 3**: Lập trình hướng đối tượng (OOP): Lớp & Đối tượng, Đóng gói (Encapsulation), Kế thừa (Inheritance), Đa hình (Polymorphism) và Trừu tượng (Abstraction).


In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc vào sys.path để import các module src
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Python version: {sys.version}')
print(f'Project root: {project_root}')


## 1. Nền Tảng Python (Chương 2)
### 1.1 Biến, Kiểu Dữ Liệu Cơ Bản & Phức Hợp


In [ ]:
# Khởi tạo các cấu trúc dữ liệu phức hợp (List, Tuple, Set, Dictionary)
order_statuses = ['completed', 'partially_refunded', 'refunded', 'cancelled'] # List
currency_pair = ('USD', 'VND') # Tuple (bất biến)
unique_payment_methods = {'card', 'paypal', 'bank_transfer'} # Set (tập hợp duy nhất)

sample_order = {
    'order_id': 'ord_001',
    'subtotal': 150.0,
    'tax': 12.0,
    'shipping': 8.0,
    'status': 'completed'
} # Dictionary

print('List of statuses:', order_statuses)
print('Unique methods:', unique_payment_methods)
print('Order sample dictionary:', sample_order)


### 1.2 Cấu Trúc Điều Khiển & Hàm


In [ ]:
def calculate_net_total(order_dict: dict, discount_rate: float = 0.0) -> float:
    """Tính tổng tiền thanh toán sau khi trừ chiết khấu và cộng thuế + phí vận chuyển."""
    if discount_rate < 0 or discount_rate > 1:
        raise ValueError('Tỷ lệ chiết khấu phải nằm trong khoảng [0, 1]')
    
    subtotal = order_dict.get('subtotal', 0.0)
    tax = order_dict.get('tax', 0.0)
    shipping = order_dict.get('shipping', 0.0)
    
    discount_amount = subtotal * discount_rate
    net_total = (subtotal - discount_amount) + tax + shipping
    return round(net_total, 2)

total = calculate_net_total(sample_order, discount_rate=0.1)
print(f'Tổng tiền sau giảm giá 10%: ${total}')


### 1.3 Xử Lý Ngoại Lệ & Đọc File An Toàn


In [ ]:
from src.core.exceptions import DataLoadError

def safe_read_first_line(filepath: Path) -> str:
    """Đọc dòng đầu tiên an toàn sử dụng context manager with và try-except-finally."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.readline().strip()
    except FileNotFoundError as e:
        raise DataLoadError(str(filepath), 'File không tồn tại trên hệ thống.') from e
    finally:
        # Khối finally luôn được thực thi
        pass

customers_csv = project_root / 'data' / 'raw' / 'customers.csv'
header = safe_read_first_line(customers_csv)
print('Header của customers.csv:', header)


## 2. Lập Trình Hướng Đối Tượng (OOP - Chương 3)
### 2.1 Domain Entities: Customer, Product, Order (Encapsulation, Methods, Dunder)


In [ ]:
from src.data_loaders.models import Customer, Product, Order

# Khởi tạo đối tượng Customer với cơ chế đóng gói (property/setter)
customer = Customer(
    customer_id='cust_9999',
    country_code='VN',
    acquisition_channel='social_media',
    customer_segment='enterprise'
)

print(repr(customer))
print(f'Customer {customer.id} is enterprise: {customer.is_enterprise()}')

# Khởi tạo Product và tính toán biên lợi nhuận
product = Product(
    product_id='prod_001',
    sku='LAPTOP-PRO-15',
    product_name='Ultrabook Pro 15',
    category='Electronics',
    unit_cost=800.0,
    list_price=1250.0
)
print(f'Sản phẩm: {product.product_name}')
print(f'Lợi nhuận gộp: ${product.profit_margin} | Tỷ lệ markup: {product.markup_percentage:.1f}%')


### 2.2 Tính Kế Thừa (Inheritance) & Trừu Tượng (Abstraction) với DataLoaders


In [ ]:
from src.data_loaders.base_loader import BaseDataLoader
from src.data_loaders.csv_loader import CSVDataLoader
from src.data_loaders.sqlite_loader import SQLiteDataLoader

# Kiểm tra quan hệ kế thừa
print('CSVDataLoader là lớp con của BaseDataLoader:', issubclass(CSVDataLoader, BaseDataLoader))
print('SQLiteDataLoader là lớp con của BaseDataLoader:', issubclass(SQLiteDataLoader, BaseDataLoader))

# Nạp dữ liệu qua CSVDataLoader
csv_loader = CSVDataLoader(project_root / 'data' / 'raw')
tables = csv_loader.load()
print('Các bảng đã nạp thành công:')
for table_name, df in tables.items():
    print(f'  - {table_name}: {len(df):,} dòng, {len(df.columns)} cột')
